# 🚗 Driver Guardian AI — SpatioTemporal ConvGRU Training Pipeline (Kaggle Edition)
## Huấn luyện Mô hình Phát hiện Ngủ gật Trực tiếp từ Video Thô qua PAFPN Backbone & ConvGRU

> **Đặc tả Kỹ thuật:**
> - **Kiến trúc mô hình**: `ConvGRUClassifier` (Bảo toàn 2D Spatial Map $40\times 40$ qua trục thời gian với cơ chế Chú ý Kép Dual Attention). Tổng cộng chuẩn **631,716 tham số**.
> - **Bộ trích xuất đặc trưng**: `ChunkedBackboneNeckExtractor` (Cắt nhỏ mini-chunk 32 frames trên GPU, trích xuất đa tỷ lệ $p_3, p_4, p_5$ từ trọng số PAFPN NMSFreeDetector).
> - **Tăng cường chuỗi video**: `VideoAugmenter` bảo toàn tính nhất quán thời gian (Temporal Consistency) qua `apply_sequence` dùng chung 1 random seed cho toàn bộ frames.
> - **Dữ liệu**: Nạp video thô (`.mp4`, `.avi`, `.mkv`), lấy mẫu đều đặn (`sample_interval=0.2s`), tensor CPU `uint8` siêu nhẹ.
> - **Tối ưu hóa**: AdamW, Cosine Annealing LR, Mixed Precision (AMP FP16), Gradient Accumulation (boundary-safe), Early Stopping.
> - **Cơ chế Khôi phục (Resume)**: Hỗ trợ tiếp tục train từ bất kỳ checkpoint nào (`epoch_N.pt`), bí danh (`'last'`, `'best'`).
> - **Môi trường**: Kaggle Notebook GPU Kernel (Tesla T4 / P100 16GB VRAM, Ubuntu Linux).

---
### 💡 Hướng dẫn Cấu hình trên Kaggle:
1. **Bật GPU**: Menu bên phải `Settings` $\rightarrow$ `Accelerator` $\rightarrow$ Chọn **GPU T4 x1** hoặc **GPU P100**.
2. **Thêm Dữ liệu (Add Data)**:
   - Thêm Dataset video ULDD đã qua tiền xử lý (chứa thư mục `train/` và `val/` hoặc manifest CSV).
   - Thêm Checkpoint NMSFreeDetector PAFPN Backbone (`best.pt`).
3. **Cập nhật đường dẫn tại Cell 4**: Kiểm tra và trỏ `dataset_dir` và `backbone_neck_checkpoint` tới đúng thư mục trong `/kaggle/input/...`.


In [1]:
# 1. Cài đặt các thư viện bổ sung nếu cần và import dependencies
import os
import sys
import time
import math
import random
import csv
import json
import shutil
import zipfile
import inspect
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Tuple, Dict, Any, Optional, Union, Sequence

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

# Import Albumentations (tự động cài đặt nếu thiếu)
try:
    import albumentations as A
except ImportError:
    import subprocess

    print("[*] Đang cài đặt thư viện albumentations...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "albumentations"])
    import albumentations as A

# Thiết lập hiển thị đồ thị inline
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print(f"[✓] PyTorch Version      : {torch.__version__}")
print(f"[✓] OpenCV Version       : {cv2.__version__}")
print(f"[✓] Albumentations       : {A.__version__}")
print(f"[✓] CUDA Sẵn sàng        : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[✓] Thiết bị GPU         : {torch.cuda.get_device_name(0)}")
    print(f"[✓] Số lượng GPU         : {torch.cuda.device_count()}")
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[✓] Tổng VRAM            : {vram_gb:.2f} GB")
else:
    print(
        "[!] CẢNH BÁO: Không phát hiện GPU. Vui lòng bật GPU Accelerator (T4 x1 hoặc P100) trong menu Kaggle Settings!")


D:\Users\tonda\anaconda3\envs\mylstm\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[✓] PyTorch Version      : 2.13.0+cu126
[✓] OpenCV Version       : 5.0.0
[✓] Albumentations       : 2.0.8
[✓] CUDA Sẵn sàng        : True
[✓] Thiết bị GPU         : NVIDIA GeForce RTX 3050 Laptop GPU
[✓] Số lượng GPU         : 1
[✓] Tổng VRAM            : 4.00 GB


In [2]:
# 2. Cố định Seed ngẫu nhiên để đảm bảo tính tái lập (Reproducibility)
def seed_everything(seed: int = 42) -> None:
    """Cố định seed cho tất cả các thư viện để kết quả huấn luyện có thể tái lập hoàn toàn."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[✓] Đã cố định toàn bộ hạt giống ngẫu nhiên: SEED = {seed}")


seed_everything(42)


[✓] Đã cố định toàn bộ hạt giống ngẫu nhiên: SEED = 42


## ⚙️ Section 2: Cấu hình Hệ thống Tập trung Duy nhất (`TrainConfig`)
Toàn bộ siêu tham số của mô hình, dữ liệu, huấn luyện và đường dẫn Kaggle được định nghĩa minh bạch trong duy nhất Cell này.
Vui lòng kiểm tra và tùy chỉnh các đường dẫn `/kaggle/input/...` cho phù hợp với Dataset bạn đã thêm vào Notebook.


In [3]:
# CELL CẤU HÌNH TẬP TRUNG DUY NHẤT (SINGLE CONFIGURATION CELL)
@dataclass
class TrainConfig:
    """
    Cấu hình toàn diện cho pipeline huấn luyện ConvGRU Drowsiness Detection trên Kaggle.
    Đồng bộ 100% với config chuẩn trong codebase và checkpoint best.pt (631,716 tham số).
    """
    # ---- 1. ĐƯỜNG DẪN DỮ LIỆU & CHECKPOINTS (KAGGLE PATHS) ----
    # Thư mục chứa dữ liệu video thô trên Kaggle (cần sửa lại cho khớp với tên dataset bạn add vào)
    dataset_dir: str = r"E:\LSTM\UL-DD_Processed"
    manifest_file: Optional[str] = "dataset_merged_split.csv"  # Hoặc None nếu nạp theo cấu trúc thư mục train/val

    # Checkpoint NMSFreeDetector PAFPN Backbone (.pt)
    backbone_neck_checkpoint: str = r"D:\Project\DATN\driver-guardian\ai\ObjectDetection_2p6M\checkpoints\231e35bb4061257f9bcb7cc5e3a0d0064e5e1beb6351a5e0adfc8d0c91ba4e11\da92589d1b67339be4491beaf147575e81a2699bbde06d654c05f852d7f0e9c9\landmark_train\best.pt"

    # Thư mục lưu đầu ra (Bắt buộc nằm trong /kaggle/working/)
    checkpoint_dir: str = "/kaggle/working/checkpoints/deepgru_raw_nmsfree"
    tb_log_dir: str = "/kaggle/working/logs/tensorboard"
    experiment_name: str = "deepgru_raw_nmsfree"
    history_csv_path: str = "/kaggle/working/logs/training_history.csv"

    # Cấu hình tiếp tục huấn luyện (Resume Training)
    enable_resume: bool = True
    resume: str = r"D:\Project\DATN\driver-guardian\ai\LSTM\checkpoints\experiments\deepgru_raw_nmsfree\epoch_5.pt"  # Đường dẫn file .pt hoặc bí danh ('last', 'best') để resume

    # ---- 2. DỮ LIỆU & DATALOADER ----
    sample_interval: float = 0.2  # Khoảng thời gian lấy mẫu (giây) -> FPS hiệu dụng = 5.0
    seq_len: Optional[int] = None  # None = dynamic padding theo độ dài clip
    image_size: Tuple[int, int] = (640, 640)
    video_exts: Tuple[str, ...] = (".avi", ".mp4", ".mkv", ".mov")
    min_frames: int = 10
    use_augmentation: bool = True  # Bật tăng cường chuỗi video đồng bộ thời gian

    batch_size: int = 2  # Batch size an toàn VRAM trên GPU 16GB
    val_batch_size: int = 4  # Batch size cho tập validation
    num_workers: int = 2  # Số tiến trình nạp dữ liệu an toàn trên Linux Kaggle
    val_num_workers: int = 2
    pin_memory: bool = False  # Tránh lỗi tràn pinned memory
    shuffle: bool = True
    seed: int = 42
    chunk_size: int = 32  # Mini-chunk chia nhỏ frames qua Backbone chống tràn VRAM

    # ---- 3. KIẾN TRÚC MÔ HÌNH (CONVGRU CLASSIFIER - 631,716 PARAMS) ----
    cnn_neck_channels: Tuple[int, int, int] = (64, 128, 256)  # Số kênh (p3, p4, p5) từ PAFPN
    adapter_dropout: float = 0.25
    input_dim: int = 128  # Chiều kênh sau SpatialReductionNeck (nén 448 -> 128)
    convgru_in_dim: int = 128
    hidden_dim: int = 64  # Số kênh ẩn ConvGRU (chuẩn hóa theo best.pt)
    convgru_hidden_dim: int = 64
    num_layers: int = 2  # Số tầng ConvGRU xếp chồng
    convgru_num_layers: int = 2
    num_classes: int = 2  # 2: Tỉnh táo (0) vs Buồn ngủ (1)
    dropout: float = 0.35
    supervision_mode: str = "attention_pooling"

    # ---- 4. HÀM MẤT MÁT (LOSS) ----
    loss_type: str = "ce"  # "ce" (CrossEntropyLoss) hoặc "bce" (BCEWithLogitsLoss)
    bce_eps: float = 1e-7
    bce_reduction: str = "mean"
    pos_weight: Optional[float] = None

    # ---- 5. TỐI ƯU HÓA (OPTIMIZER & SCHEDULER) ----
    epochs: int = 20
    lr0: float = 0.001
    lr_min_factor: float = 0.01  # lr_min = lr0 * 0.01 = 1e-5
    weight_decay: float = 0.0001
    optimizer: str = "adamw"  # "adamw" hoặc "sgd"
    betas: Tuple[float, float] = (0.9, 0.999)
    momentum: float = 0.9
    grad_clip_norm: float = 1.0  # Gradient clipping chống nổ gradient
    gradient_accumulation_steps: int = 4  # Tích lũy 4 bước -> Effective batch size = 16
    empty_cache_interval: int = 2  # Dọn cache GPU mỗi 2 epoch
    use_scheduler: bool = True  # CosineAnnealingLR

    # ---- 6. GIÁM SÁT, EARLY STOPPING & CHECKPOINTS ----
    save_all_epochs: bool = False  # False để tiết kiệm dung lượng ổ đĩa Kaggle (tối đa 20GB)
    save_ckpt_interval_epochs: int = 5
    ckpt_keep_last: Optional[int] = 3
    early_stopping: bool = True
    patience: int = 10
    monitor_metric: str = "val_f1"
    monitor_mode: str = "max"
    min_delta: float = 0.0001
    enable_step_logging: bool = True
    log_step_interval: int = 1

    # ---- 7. RUNTIME & PHẦN CỨNG ----
    device: str = "cuda"
    amp: bool = True  # Mixed Precision FP16
    val_interval_epochs: int = 1
    use_tqdm: bool = True

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


cfg = TrainConfig()
print("[✓] Đã khởi tạo cấu hình TrainConfig thành công!")
print(f"[*] Thiết bị mục tiêu : {cfg.device}")
print(
    f"[*] Batch size        : {cfg.batch_size} (Accumulation Steps: {cfg.gradient_accumulation_steps} -> Effective: {cfg.batch_size * cfg.gradient_accumulation_steps})")
print(f"[*] Tổng Epochs       : {cfg.epochs}")


[✓] Đã khởi tạo cấu hình TrainConfig thành công!
[*] Thiết bị mục tiêu : cuda
[*] Batch size        : 2 (Accumulation Steps: 4 -> Effective: 8)
[*] Tổng Epochs       : 20


## 🧠 Section 3: Bộ Trích xuất Đặc trưng Backbone PAFPN
Định nghĩa đầy đủ các khối CNN cơ sở và kiến trúc mạng ghép nối `BackboneNeck` (PAFPN từ NMSFreeDetector) phục vụ trích xuất đặc trưng $(p_3, p_4, p_5)$.
Phần đầu dự đoán `DetectHead` của bài toán Object Detection được loại bỏ để đảm bảo tính tối giản, độc lập và nhẹ tải.


In [4]:
# 1. Các khối tích chập chuẩn hóa của NMSFreeDetector PAFPN
def autopad(k, p=None, d=1):
    """Tự động tính toán padding để bảo toàn kích thước không gian."""
    if p is not None:
        return p
    return (d * (k - 1) + 1) // 2


class Conv(nn.Module):
    def __init__(self, c1, c2, k=1, s=1, p=None, g=1, d=1, act=True):
        super().__init__()
        self.conv = nn.Conv2d(c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Bottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=0.5):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = Conv(c1, c_, 3, 1)
        self.cv2 = Conv(c_, c2, 3, 1)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        y = self.cv2(self.cv1(x))
        return x + y if self.add else y


class C2f(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=True, e=0.5):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv((2 + n) * self.c, c2, 1, 1)
        self.m = nn.ModuleList(Bottleneck(self.c, self.c, shortcut, e=1.0) for _ in range(n))

    def forward(self, x):
        y = list(self.cv1(x).chunk(2, 1))
        for m in self.m:
            y.append(m(y[-1]))
        return self.cv2(torch.cat(y, 1))


class CIB(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=0.5, large_kernel=False):
        super().__init__()
        c_ = int(c2 * e)
        k = 7 if large_kernel else 3
        self.block = nn.Sequential(
            Conv(c1, c1, k, 1, g=c1),
            Conv(c1, 2 * c_, 1, 1),
            Conv(2 * c_, 2 * c_, k, 1, g=2 * c_),
            Conv(2 * c_, c2, 1, 1),
            Conv(c2, c2, k, 1, g=c2),
        )
        self.add = shortcut and c1 == c2

    def forward(self, x):
        y = self.block(x)
        return x + y if self.add else y


class C2fCIB(C2f):
    def __init__(self, c1, c2, n=1, shortcut=False, e=0.5, large_kernel=False):
        super().__init__(c1, c2, n, shortcut, e)
        self.m = nn.ModuleList(CIB(self.c, self.c, shortcut, e=1.0, large_kernel=large_kernel) for _ in range(n))


class SPPF(nn.Module):
    def __init__(self, c1, c2, k=5):
        super().__init__()
        c_ = c1 // 2
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(4 * c_, c2, 1, 1)
        self.m = nn.MaxPool2d(k, stride=1, padding=k // 2)

    def forward(self, x):
        x = self.cv1(x)
        y1 = self.m(x)
        y2 = self.m(y1)
        y3 = self.m(y2)
        return self.cv2(torch.cat((x, y1, y2, y3), 1))


class Attention(nn.Module):
    def __init__(self, dim, num_heads=4, mlp_ratio=2.0, layer_scale=1e-2):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        hidden_dim = int(dim * mlp_ratio)
        self.qkv = nn.Conv2d(dim, 3 * dim, 1, bias=False)
        self.proj = Conv(dim, dim, 1, 1, act=False)
        self.pe = Conv(dim, dim, 3, 1, g=dim, act=False)
        self.ffn = nn.Sequential(
            Conv(dim, hidden_dim, 1, 1),
            Conv(hidden_dim, dim, 1, 1, act=False),
        )
        self.gamma1 = nn.Parameter(layer_scale * torch.ones(dim))
        self.gamma2 = nn.Parameter(layer_scale * torch.ones(dim))

    def forward(self, x):
        B, C, H, W = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, H * W)
        q, k, v = qkv.permute(1, 0, 2, 4, 3).contiguous().unbind(0)
        v_spatial = v.transpose(-2, -1).reshape(B, C, H, W)
        out = F.scaled_dot_product_attention(q, k, v, dropout_p=0.0)
        out = out.transpose(-2, -1).reshape(B, C, H, W)
        out = self.proj(out + self.pe(v_spatial))
        x = x + self.gamma1.to(out.dtype).view(1, -1, 1, 1) * out
        out = self.ffn(x)
        return x + self.gamma2.to(out.dtype).view(1, -1, 1, 1) * out


class C2fPSA(nn.Module):
    def __init__(self, c1, c2, n=1, e=0.5, num_heads=4):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = Conv(c1, 2 * self.c, 1, 1)
        self.cv2 = Conv(2 * self.c, c2, 1, 1)
        self.m = nn.Sequential(*(Attention(self.c, num_heads=num_heads) for _ in range(n)))

    def forward(self, x):
        a, b = self.cv1(x).chunk(2, 1)
        return self.cv2(torch.cat((a, self.m(b)), 1))


class SCDown(nn.Module):
    def __init__(self, c1, c2, k=3, s=1, p=None, d=1, act=True):
        super().__init__()
        self.cv1 = Conv(c1, c2, 1, 1, act=act)
        self.cv2 = Conv(c2, c2, k, s, p=p, g=c2, d=d, act=False)

    def forward(self, x):
        return self.cv2(self.cv1(x))


class Backbone(nn.Module):
    def __init__(self, w=(16, 32, 64, 128, 256), n=(1, 2, 2, 1)):
        super().__init__()
        c0, c1, c2, c3, c4 = w
        n0, n1, n2, n3 = n
        self.stem = Conv(3, c0, 3, 2)
        self.stage1 = nn.Sequential(Conv(c0, c1, 3, 2), C2f(c1, c1, n=n0, shortcut=True))
        self.stage2 = nn.Sequential(Conv(c1, c2, 3, 2), C2f(c2, c2, n=n1, shortcut=True))
        self.stage3 = nn.Sequential(SCDown(c2, c3, 3, 2), C2fCIB(c3, c3, n=n2, shortcut=True))
        self.stage4 = nn.Sequential(
            SCDown(c3, c4, 3, 2),
            C2fCIB(c4, c4, n=n3, shortcut=True, large_kernel=True),
            SPPF(c4, c4, k=5),
            C2fPSA(c4, c4, n=2, e=0.5)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        p3 = self.stage2(x)
        p4 = self.stage3(p3)
        p5 = self.stage4(p4)
        return p3, p4, p5


class PAFPN(nn.Module):
    def __init__(self, chs=(64, 128, 256), n=1):
        super().__init__()
        c3, c4, c5 = chs
        self.up = nn.Upsample(scale_factor=2, mode="nearest")
        # Top-down
        self.c2f_p4 = C2f(c5 + c4, c4, n=n, shortcut=True)
        self.c2f_p3 = C2f(c4 + c3, c3, n=n, shortcut=False)
        # Bottom-up
        self.down3 = Conv(c3, c3, 3, 2)
        self.c2f_n4 = C2fCIB(c3 + c4, c4, n=n, shortcut=True)
        self.down4 = SCDown(c4, c4, 3, 2)
        self.c2f_n5 = C2fCIB(c4 + c5, c5, n=n, shortcut=True, large_kernel=True)

    def forward(self, p3, p4, p5):
        # Top-down
        x = self.up(p5)
        x = torch.cat([x, p4], 1)
        p4_td = self.c2f_p4(x)

        x = self.up(p4_td)
        x = torch.cat([x, p3], 1)
        p3_out = self.c2f_p3(x)

        # Bottom-up
        x = self.down3(p3_out)
        x = torch.cat([x, p4_td], 1)
        p4_out = self.c2f_n4(x)

        x = self.down4(p4_out)
        x = torch.cat([x, p5], 1)
        p5_out = self.c2f_n5(x)

        return p3_out, p4_out, p5_out


class BackboneNeck(nn.Module):
    """Module tổ hợp gồm Backbone và PAFPN Neck trích xuất (p3, p4, p5)."""

    def __init__(self, backbone: nn.Module, neck: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.neck = neck

    def forward(self, x):
        return self.neck(*self.backbone(x))


print("[✓] Đã định nghĩa các khối CNN và kiến trúc BackboneNeck thành công!")


[✓] Đã định nghĩa các khối CNN và kiến trúc BackboneNeck thành công!


In [5]:
# 2. Lớp trích xuất đặc trưng Mini-Chunk trên GPU (ChunkedBackboneNeckExtractor)
class ChunkedBackboneNeckExtractor(nn.Module):
    """
    Trích xuất đặc trưng đa tầng (p3, p4, p5) trực tiếp trên GPU theo cơ chế Mini-Chunk.
    Chia nhỏ video thành các chunk (chunk_size=32) để bảo đảm không bao giờ vượt trần VRAM.
    Toàn bộ trọng số được đóng băng (requires_grad=False).
    """

    def __init__(
            self,
            checkpoint_path: Union[str, Path],
            device: str = "cuda",
            chunk_size: int = 32,
            use_fp16: bool = True,
            img_size: int = 640
    ) -> None:
        super().__init__()
        self.checkpoint_path = Path(checkpoint_path)
        self.device = torch.device(device if torch.cuda.is_available() else "cpu")
        self.chunk_size = max(1, chunk_size)
        self.use_fp16 = use_fp16
        self.img_size = img_size
        self.model = self._load_model()

    def _load_model(self) -> nn.Module:
        # Cấu hình chuẩn NMSFreeDetector PAFPN Backbone (2.6M)
        backbone_w = (16, 32, 64, 128, 256)
        backbone_n = (1, 2, 2, 1)
        neck_n = 1

        checkpoint = None
        if self.checkpoint_path.exists():
            print(f"[*] Nạp checkpoint Backbone từ: {self.checkpoint_path}")
            try:
                checkpoint = torch.load(self.checkpoint_path, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"[!] Lỗi khi nạp checkpoint Backbone: {e}")
        else:
            print(f"[CẢNH BÁO] Không tìm thấy file checkpoint tại: {self.checkpoint_path}")
            print("[*] Khởi tạo BackboneNeck với trọng số ngẫu nhiên (chế độ thử nghiệm).")

        # Đồng bộ từ metadata nếu có trong checkpoint
        if checkpoint and isinstance(checkpoint, dict) and "metadata" in checkpoint:
            meta_arch = checkpoint["metadata"].get("architecture", {})
            if "trunk_backbone_w" in meta_arch:
                backbone_w = tuple(meta_arch["trunk_backbone_w"])
            if "trunk_backbone_n" in meta_arch:
                backbone_n = tuple(meta_arch["trunk_backbone_n"])
            if "trunk_neck_n" in meta_arch:
                neck_n = int(meta_arch["trunk_neck_n"])

        c3, c4, c5 = backbone_w[2], backbone_w[3], backbone_w[4]
        backbone = Backbone(w=backbone_w, n=backbone_n)
        neck = PAFPN(chs=(c3, c4, c5), n=neck_n)
        backbone_neck = BackboneNeck(backbone, neck).eval()

        if checkpoint:
            state_dict = checkpoint.get("ema") or checkpoint.get("model") or checkpoint
            # Lọc chỉ lấy các trọng số thuộc backbone và neck, bỏ qua head
            filtered_state = {k: v for k, v in state_dict.items() if k.startswith(("backbone.", "neck."))}
            missing, unexpected = backbone_neck.load_state_dict(filtered_state, strict=False)
            print(f"[✓] Đã nạp thành công {len(filtered_state)} tensors trọng số vào BackboneNeck.")

        # Đóng băng 100% tham số để tiết kiệm VRAM và giải phóng gradient graph
        for param in backbone_neck.parameters():
            param.requires_grad = False

        backbone_neck = backbone_neck.to(self.device)
        return backbone_neck

    def forward(self, batch_frames: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Trích xuất đặc trưng theo cơ chế Mini-Chunking.
        batch_frames: Tensor ảnh uint8 hoặc float32 dạng [B, T, 3, H, W] trên GPU.
        Returns: Tuple (p3, p4, p5) với dạng [B, T, C, H, W] trên GPU.
        """
        B, T, C, H, W = batch_frames.shape
        flat_frames = batch_frames.view(B * T, C, H, W)
        p3_list, p4_list, p5_list = [], [], []

        with torch.inference_mode():
            for i in range(0, B * T, self.chunk_size):
                chunk = flat_frames[i: i + self.chunk_size]
                # Ép kiểu [0, 255] uint8 -> [0.0, 1.0] float32 TỪNG CHUNK MỘT để tiết kiệm bộ nhớ
                if chunk.dtype == torch.uint8:
                    chunk = chunk.float() / 255.0

                if self.use_fp16 and self.device.type == "cuda":
                    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                        out3, out4, out5 = self.model(chunk)
                else:
                    out3, out4, out5 = self.model(chunk)

                # Ép kiểu float32 tương thích hoàn hảo với ConvGRU
                p3_list.append(out3.float())
                p4_list.append(out4.float())
                p5_list.append(out5.float())

        p3_flat = torch.cat(p3_list, dim=0)
        p4_flat = torch.cat(p4_list, dim=0)
        p5_flat = torch.cat(p5_list, dim=0)

        p3 = p3_flat.view(B, T, *p3_flat.shape[1:])
        p4 = p4_flat.view(B, T, *p4_flat.shape[1:])
        p5 = p5_flat.view(B, T, *p5_flat.shape[1:])
        return p3, p4, p5


print("[✓] Đã định nghĩa lớp ChunkedBackboneNeckExtractor thành công!")


[✓] Đã định nghĩa lớp ChunkedBackboneNeckExtractor thành công!


## 🤖 Section 4: Kiến trúc Không gian - Thời gian `ConvGRUClassifier`
Định nghĩa kiến trúc mô hình chính bảo toàn đặc trưng không gian 2D ($40\times 40$) qua chuỗi thời gian:
1. `SpatialReductionNeck`: Căn chỉnh $p_3$ ($80\times 80 \to 40\times 40$ qua MaxPool), $p_4$ ($40\times 40$), $p_5$ ($20\times 20 \to 40\times 40$ qua Upsample), ghép kênh $448 \to 128$ kênh.
2. `ConvGRUCell` & `ConvGRU`: Lan truyền thông tin không gian - thời gian qua các bước lặp với Kernel Fusion.
3. `SpatialAttentionPooling` & `TemporalAttentionPooling`: Cơ chế chú ý kép (Dual Attention) tập trung vào khuôn mặt và loại bỏ nhiễu padding.
4. `ConvGRUClassifier`: Khớp chuẩn xác **631,716 tham số** tương thích 100% với checkpoint `best.pt`.


In [6]:
# 1. Cổ giảm kênh không gian (SpatialReductionNeck)
class SpatialReductionNeck(nn.Module):
    """
    Cổ Giảm Kênh Không Gian Đa Tỷ Lệ (Multi-Scale Spatial Reduction Neck).
    Căn chỉnh các tầng đặc trưng từ NMSFreeDetector PAFPN:
      - p3: [B, (T), 64, 80, 80]  -> MaxPool2d(s=2) -> 40x40
      - p4: [B, (T), 128, 40, 40] -> Giữ nguyên     -> 40x40
      - p5: [B, (T), 256, 20, 20] -> Upsample(x2)   -> 40x40
    Nén 448 -> out_channels (128) qua Conv2d 1x1 + BN + SiLU.
    """

    def __init__(
            self,
            in_channels: Union[Tuple[Tuple[int, ...], ...], Tuple[int, ...], Sequence[int]] = (64, 128, 256),
            out_channels: int = 128,
            dropout: float = 0.25,
            target_size: Tuple[int, int] = (40, 40)
    ):
        super().__init__()
        if len(in_channels) > 0 and isinstance(in_channels[0], (tuple, list)):
            self.channel_dims = tuple(c[0] for c in in_channels)
        else:
            self.channel_dims = tuple(int(c) for c in in_channels)

        self.num_scales = len(self.channel_dims)
        self.total_in_channels = sum(self.channel_dims)
        self.out_channels = int(out_channels)
        self.target_size = tuple(target_size)

        self.p3_down = nn.MaxPool2d(kernel_size=2, stride=2)
        self.p5_up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

        self.reduction = nn.Sequential(
            nn.Conv2d(
                self.total_in_channels,
                self.out_channels,
                kernel_size=1,
                stride=1,
                padding=0,
                bias=False
            ),
            nn.BatchNorm2d(self.out_channels),
            nn.SiLU(inplace=True),
            nn.Dropout2d(p=dropout) if dropout > 0.0 else nn.Identity()
        )

    def _align_features(
            self,
            p3: torch.Tensor,
            p4: torch.Tensor,
            p5: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        th, tw = self.target_size
        out_p3 = self.p3_down(p3) if p3.shape[-2:] != (th, tw) else p3
        out_p4 = p4 if p4.shape[-2:] == (th, tw) else F.interpolate(p4, size=(th, tw), mode="bilinear",
                                                                    align_corners=False)
        out_p5 = self.p5_up(p5) if p5.shape[-2:] != (th, tw) else p5
        return out_p3, out_p4, out_p5

    def forward(
            self,
            features: Union[Tuple[torch.Tensor, ...], List[torch.Tensor], torch.Tensor],
            seq_lens: Optional[torch.Tensor] = None,
            *args: torch.Tensor
    ) -> torch.Tensor:
        if len(args) > 0:
            features = (features, *args)

        if isinstance(features, (tuple, list)):
            if len(features) != self.num_scales:
                raise ValueError(
                    f"[SpatialReductionNeck] Yêu cầu {self.num_scales} tầng đặc trưng, nhưng nhận {len(features)}."
                )
            p3, p4, p5 = features[0], features[1], features[2]
            is_5d = (p3.dim() == 5)
            orig_b, orig_t = 0, 0

            if is_5d:
                orig_b, orig_t = p3.shape[0], p3.shape[1]
                p3 = p3.reshape(orig_b * orig_t, *p3.shape[2:])
                p4 = p4.reshape(orig_b * orig_t, *p4.shape[2:])
                p5 = p5.reshape(orig_b * orig_t, *p5.shape[2:])

            out_p3, out_p4, out_p5 = self._align_features(p3, p4, p5)
            fused = torch.cat([out_p3, out_p4, out_p5], dim=1)  # [N, total_in, 40, 40]

            if is_5d and seq_lens is not None:
                time_indices = torch.arange(orig_t, device=fused.device).unsqueeze(0)
                lens_expanded = seq_lens.to(fused.device).unsqueeze(1)
                mask = (time_indices < lens_expanded).view(-1)

                valid_fused = fused[mask]
                if valid_fused.numel() > 0:
                    valid_reduced = self.reduction(valid_fused)
                else:
                    valid_reduced = torch.empty((0, self.out_channels, self.target_size[0], self.target_size[1]),
                                                device=fused.device, dtype=fused.dtype)

                reduced = torch.zeros((orig_b * orig_t, self.out_channels, self.target_size[0], self.target_size[1]),
                                      device=fused.device, dtype=fused.dtype)
                reduced[mask] = valid_reduced.to(reduced.dtype)
            else:
                reduced = self.reduction(fused)

            if is_5d:
                reduced = reduced.view(orig_b, orig_t, self.out_channels, *reduced.shape[-2:])
            return reduced

        elif isinstance(features, torch.Tensor):
            feat = features
            is_5d = (feat.dim() == 5)
            orig_b, orig_t = 0, 0

            if is_5d:
                orig_b, orig_t = feat.shape[0], feat.shape[1]
                feat = feat.reshape(orig_b * orig_t, *feat.shape[2:])

            if feat.shape[1] == self.total_in_channels:
                if is_5d and seq_lens is not None:
                    time_indices = torch.arange(orig_t, device=feat.device).unsqueeze(0)
                    lens_expanded = seq_lens.to(feat.device).unsqueeze(1)
                    mask = (time_indices < lens_expanded).view(-1)

                    valid_feat = feat[mask]
                    if valid_feat.numel() > 0:
                        valid_reduced = self.reduction(valid_feat)
                    else:
                        valid_reduced = torch.empty((0, self.out_channels, self.target_size[0], self.target_size[1]),
                                                    device=feat.device, dtype=feat.dtype)

                    reduced = torch.zeros(
                        (orig_b * orig_t, self.out_channels, self.target_size[0], self.target_size[1]),
                        device=feat.device, dtype=feat.dtype)
                    reduced[mask] = valid_reduced.to(reduced.dtype)
                else:
                    reduced = self.reduction(feat)
            elif feat.shape[1] == self.out_channels:
                reduced = feat
            else:
                raise ValueError(
                    f"[SpatialReductionNeck] Số kênh đầu vào ({feat.shape[1]}) không khớp "
                    f"total_in ({self.total_in_channels}) hoặc out_channels ({self.out_channels})."
                )

            if is_5d:
                reduced = reduced.view(orig_b, orig_t, self.out_channels, *reduced.shape[-2:])
            return reduced
        else:
            raise TypeError(f"[SpatialReductionNeck] Kiểu dữ liệu không hợp lệ: {type(features)}")


# 2. Tế bào và Tầng Convolutional GRU (ConvGRUCell & ConvGRU)
class ConvGRUCell(nn.Module):
    """
    Tế bào Convolutional GRU đơn tầng với Kernel Fusion (gộp Reset và Update Gate).
    """

    def __init__(
            self,
            input_dim: int,
            hidden_dim: int,
            kernel_size: Union[int, Tuple[int, int]] = 3,
            bias: bool = True,
            padding_mode: str = "zeros"
    ):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_dim = int(hidden_dim)
        self.bias = bias

        if isinstance(kernel_size, int):
            self.kernel_size = (kernel_size, kernel_size)
        else:
            self.kernel_size = tuple(kernel_size)

        self.padding = (self.kernel_size[0] // 2, self.kernel_size[1] // 2)

        self.conv_gates = nn.Conv2d(
            in_channels=self.input_dim + self.hidden_dim,
            out_channels=2 * self.hidden_dim,
            kernel_size=self.kernel_size,
            padding=self.padding,
            padding_mode=padding_mode,
            bias=self.bias
        )
        self.conv_candidate = nn.Conv2d(
            in_channels=self.input_dim + self.hidden_dim,
            out_channels=self.hidden_dim,
            kernel_size=self.kernel_size,
            padding=self.padding,
            padding_mode=padding_mode,
            bias=self.bias
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        with torch.no_grad():
            w_gates = self.conv_gates.weight
            w_gates_x = w_gates[:, :self.input_dim, :, :]
            w_gates_h = w_gates[:, self.input_dim:, :, :]

            nn.init.kaiming_normal_(w_gates_x, mode="fan_out", nonlinearity="relu")
            nn.init.orthogonal_(w_gates_h)

            if self.conv_gates.bias is not None:
                nn.init.zeros_(self.conv_gates.bias)
                self.conv_gates.bias[self.hidden_dim:].fill_(1.0)

            w_can = self.conv_candidate.weight
            w_can_x = w_can[:, :self.input_dim, :, :]
            w_can_h = w_can[:, self.input_dim:, :, :]

            nn.init.kaiming_normal_(w_can_x, mode="fan_out", nonlinearity="tanh")
            nn.init.orthogonal_(w_can_h)

            if self.conv_candidate.bias is not None:
                nn.init.zeros_(self.conv_candidate.bias)

    def init_hidden(
            self,
            batch_size: int,
            spatial_size: Tuple[int, int],
            device: Optional[torch.device] = None,
            dtype: Optional[torch.dtype] = None
    ) -> torch.Tensor:
        target_device = device if device is not None else self.conv_gates.weight.device
        target_dtype = dtype if dtype is not None else self.conv_gates.weight.dtype

        return torch.zeros(
            batch_size,
            self.hidden_dim,
            spatial_size[0],
            spatial_size[1],
            device=target_device,
            dtype=target_dtype
        )

    def forward(
            self,
            input_tensor: torch.Tensor,
            h_cur: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        if input_tensor.dim() != 4:
            raise ValueError(f"[ConvGRUCell] Yêu cầu input_tensor 4D [B, C, H, W], nhận {list(input_tensor.shape)}.")

        batch_size, _, height, width = input_tensor.shape

        if h_cur is None:
            h_cur = self.init_hidden(
                batch_size=batch_size,
                spatial_size=(height, width),
                device=input_tensor.device,
                dtype=input_tensor.dtype
            )

        combined = torch.cat([input_tensor, h_cur], dim=1)
        gates = self.conv_gates(combined)
        reset_raw, update_raw = torch.split(gates, self.hidden_dim, dim=1)

        reset_gate = torch.sigmoid(reset_raw)
        update_gate = torch.sigmoid(update_raw)

        h_cur_casted = h_cur.to(dtype=reset_gate.dtype)
        in_casted = input_tensor.to(dtype=reset_gate.dtype)

        combined_candidate = torch.cat([in_casted, reset_gate * h_cur_casted], dim=1)
        candidate_raw = self.conv_candidate(combined_candidate)
        candidate = torch.tanh(candidate_raw)

        h_next = (1.0 - update_gate) * h_cur_casted + update_gate * candidate
        return h_next


class ConvGRU(nn.Module):
    """
    Module Convolutional GRU Đa Tầng bảo toàn lưới không gian 2D.
    """

    def __init__(
            self,
            input_dim: int,
            hidden_dim: Union[int, List[int], Tuple[int, ...]],
            kernel_size: Union[int, Tuple[int, int], List[Union[int, Tuple[int, int]]]] = 3,
            num_layers: int = 1,
            batch_first: bool = True,
            bias: bool = True,
            return_all_layers: bool = False,
            dropout: float = 0.0,
            residual: bool = False
    ):
        super().__init__()
        self.input_dim = int(input_dim)
        self.num_layers = int(num_layers)
        self.batch_first = batch_first
        self.bias = bias
        self.return_all_layers = return_all_layers
        self.residual = residual

        if isinstance(hidden_dim, (list, tuple)):
            self.hidden_dims = [int(h) for h in hidden_dim]
        else:
            self.hidden_dims = [int(hidden_dim)] * self.num_layers

        if isinstance(kernel_size, list):
            self.kernel_sizes = kernel_size
        else:
            self.kernel_sizes = [kernel_size] * self.num_layers

        cell_list: List[nn.Module] = []
        dropout_list: List[nn.Module] = []

        for layer_idx in range(self.num_layers):
            cur_in_dim = self.input_dim if layer_idx == 0 else self.hidden_dims[layer_idx - 1]
            cur_hid_dim = self.hidden_dims[layer_idx]
            cur_k_size = self.kernel_sizes[layer_idx]

            cell = ConvGRUCell(
                input_dim=cur_in_dim,
                hidden_dim=cur_hid_dim,
                kernel_size=cur_k_size,
                bias=self.bias
            )
            cell_list.append(cell)

            if dropout > 0.0 and layer_idx < self.num_layers - 1:
                dropout_list.append(nn.Dropout2d(p=dropout))
            else:
                dropout_list.append(nn.Identity())

        self.cells = nn.ModuleList(cell_list)
        self.dropouts = nn.ModuleList(dropout_list)

    def init_hidden(
            self,
            batch_size: int,
            spatial_size: Tuple[int, int],
            device: Optional[torch.device] = None,
            dtype: Optional[torch.dtype] = None
    ) -> List[torch.Tensor]:
        return [
            cell.init_hidden(batch_size, spatial_size, device=device, dtype=dtype)
            for cell in self.cells
        ]

    def forward(
            self,
            input_tensor: torch.Tensor,
            hidden_state: Optional[List[torch.Tensor]] = None,
            seq_lens: Optional[torch.Tensor] = None
    ) -> Tuple[Union[torch.Tensor, List[torch.Tensor]], List[torch.Tensor]]:
        if not self.batch_first:
            input_tensor = input_tensor.permute(1, 0, 2, 3, 4)

        batch_size, seq_len, in_channels, height, width = input_tensor.shape

        if hidden_state is None:
            cur_states = self.init_hidden(
                batch_size=batch_size,
                spatial_size=(height, width),
                device=input_tensor.device,
                dtype=input_tensor.dtype
            )
        else:
            cur_states = list(hidden_state)

        masks: Optional[torch.Tensor] = None
        if seq_lens is not None:
            time_indices = torch.arange(seq_len, device=input_tensor.device).unsqueeze(0)
            seq_lens_tensor = seq_lens.to(input_tensor.device).unsqueeze(1)
            masks = (time_indices < seq_lens_tensor)

        layer_outputs: List[torch.Tensor] = []
        last_states: List[torch.Tensor] = []
        cur_layer_input = input_tensor

        for layer_idx in range(self.num_layers):
            cell = self.cells[layer_idx]
            dropout_layer = self.dropouts[layer_idx]
            h_t = cur_states[layer_idx]

            time_outputs: List[torch.Tensor] = []

            for t in range(seq_len):
                x_t = cur_layer_input[:, t, :, :, :]
                h_next = cell(x_t, h_t)

                if masks is not None:
                    mask_t = masks[:, t].view(batch_size, 1, 1, 1).to(dtype=h_next.dtype)
                    h_t = mask_t * h_next + (1.0 - mask_t) * h_t.to(dtype=h_next.dtype)
                else:
                    h_t = h_next

                time_outputs.append(h_t)

            layer_output = torch.stack(time_outputs, dim=1)

            if self.residual and cur_layer_input.shape[2] == layer_output.shape[2]:
                layer_output = layer_output + cur_layer_input.to(dtype=layer_output.dtype)

            if not isinstance(dropout_layer, nn.Identity):
                b_sz, t_sz, c_sz, h_sz, w_sz = layer_output.shape
                layer_output_reshaped = layer_output.view(b_sz * t_sz, c_sz, h_sz, w_sz)
                layer_output = dropout_layer(layer_output_reshaped).view(b_sz, t_sz, c_sz, h_sz, w_sz)

            cur_layer_input = layer_output
            layer_outputs.append(layer_output)
            last_states.append(h_t)

        if not self.batch_first:
            layer_outputs = [out.permute(1, 0, 2, 3, 4) for out in layer_outputs]

        final_output = layer_outputs if self.return_all_layers else layer_outputs[-1]
        return final_output, last_states


# 3. Cơ chế Chú ý Không gian & Thời gian (Dual Attention Pooling)
class SpatialAttentionPooling(nn.Module):
    """
    Gom tụ không gian có trọng số chú ý 2D (Spatial Attention Pooling).
    Học bản đồ chú ý trên lưới không gian 40x40 qua Conv 3x3 -> BN -> SiLU -> Conv 1x1.
    """

    def __init__(self, in_channels: int, hidden_dim: Optional[int] = None):
        super().__init__()
        self.in_channels = int(in_channels)
        mid_channels = hidden_dim or max(16, self.in_channels // 2)

        self.attn_conv = nn.Sequential(
            nn.Conv2d(self.in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(mid_channels, 1, kernel_size=1, bias=True)
        )

    def forward(
            self,
            x: torch.Tensor,
            seq_lens: Optional[torch.Tensor] = None,
            return_weights: bool = False
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        is_5d = (x.dim() == 5)
        orig_b, orig_t = 0, 0

        if is_5d:
            orig_b, orig_t, c, h, w = x.shape
            x_reshaped = x.reshape(orig_b * orig_t, c, h, w)
        elif x.dim() == 4:
            x_reshaped = x
            _, c, h, w = x.shape
        else:
            raise ValueError(f"[SpatialAttentionPooling] Yêu cầu tensor 4D hoặc 5D, nhận {x.dim()}D.")

        if is_5d and seq_lens is not None:
            time_indices = torch.arange(orig_t, device=x.device).unsqueeze(0)
            lens_expanded = seq_lens.to(x.device).unsqueeze(1)
            mask = (time_indices < lens_expanded).view(-1)

            valid_x = x_reshaped[mask]
            if valid_x.numel() > 0:
                valid_scores = self.attn_conv(valid_x)
            else:
                valid_scores = torch.empty((0, 1, h, w), device=x.device, dtype=x.dtype)

            scores = torch.full((orig_b * orig_t, 1, h, w), fill_value=-1e9 if x.dtype == torch.float32 else -1e4,
                                device=x.device, dtype=x.dtype)
            scores[mask] = valid_scores.to(scores.dtype)
        else:
            scores = self.attn_conv(x_reshaped)

        scores_flat = scores.view(-1, 1, h * w)
        weights_flat = F.softmax(scores_flat, dim=-1)
        weights = weights_flat.view(-1, 1, h, w)

        pooled = (x_reshaped * weights).sum(dim=(-2, -1))

        if is_5d:
            pooled = pooled.view(orig_b, orig_t, c)
            weights = weights.view(orig_b, orig_t, 1, h, w)

        if return_weights:
            return pooled, weights
        return pooled


class TemporalAttentionPooling(nn.Module):
    """
    Gom tụ chuỗi thời gian có trọng số chú ý động (Temporal Attention MLP).
    """

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.hidden_dim = int(hidden_dim)
        self.attn = nn.Sequential(
            nn.Linear(self.hidden_dim, max(16, self.hidden_dim // 2)),
            nn.Tanh(),
            nn.Linear(max(16, self.hidden_dim // 2), 1)
        )

    def forward(
            self,
            seq_features: torch.Tensor,
            seq_lens: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        b, t, h = seq_features.shape
        scores = self.attn(seq_features).squeeze(-1)

        if seq_lens is not None:
            time_indices = torch.arange(t, device=seq_features.device).unsqueeze(0)
            lens_expanded = seq_lens.to(seq_features.device).unsqueeze(1)
            mask = (time_indices < lens_expanded)
            fill_value = -1e4 if scores.dtype == torch.float16 else -1e9
            scores = scores.masked_fill(~mask, fill_value)

        weights = F.softmax(scores, dim=-1)
        pooled = torch.bmm(weights.unsqueeze(1), seq_features).squeeze(1)
        return pooled, weights


# 4. Mô hình Hoàn chỉnh ConvGRUClassifier
class ConvGRUClassifier(nn.Module):
    """
    Mô hình Phân Loại Chuỗi Không Gian - Thời Gian Spatio-Temporal ConvGRU.
    Luồng xử lý:
      (p3, p4, p5) -> SpatialReductionNeck -> ConvGRU (2D) -> SpatialAttentionPooling -> TemporalAttentionPooling -> FC Head.
    """

    def __init__(
            self,
            input_dim: int = 128,
            hidden_dim: Union[int, List[int], Tuple[int, ...]] = 64,
            num_layers: int = 2,
            kernel_size: Union[int, Tuple[int, int]] = 3,
            num_classes: int = 2,
            spatial_in_channels: Tuple[int, ...] = (64, 128, 256),
            neck_dropout: float = 0.25,
            convgru_dropout: float = 0.1,
            dropout: float = 0.35,
            supervision_mode: str = "attention_pooling"
    ):
        super().__init__()
        self.input_dim = int(input_dim)
        self.num_layers = int(num_layers)
        self.num_classes = int(num_classes)
        self.supervision_mode = supervision_mode.lower()

        if isinstance(hidden_dim, (list, tuple)):
            self.hidden_dims = [int(h) for h in hidden_dim]
        else:
            self.hidden_dims = [int(hidden_dim)] * self.num_layers
        self.last_hidden_dim = self.hidden_dims[-1]

        self.spatial_neck = SpatialReductionNeck(
            in_channels=spatial_in_channels,
            out_channels=self.input_dim,
            dropout=neck_dropout
        )

        self.convgru = ConvGRU(
            input_dim=self.input_dim,
            hidden_dim=self.hidden_dims,
            kernel_size=kernel_size,
            num_layers=self.num_layers,
            batch_first=True,
            bias=True,
            return_all_layers=False,
            dropout=convgru_dropout,
            residual=False
        )

        self.spatial_pooling = SpatialAttentionPooling(
            in_channels=self.last_hidden_dim
        )

        self.temporal_pooling = TemporalAttentionPooling(
            hidden_dim=self.last_hidden_dim
        )

        self.fc_out = nn.Sequential(
            nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity(),
            nn.Linear(self.last_hidden_dim, self.num_classes)
        )

    @classmethod
    def from_config(cls, config: Any) -> "ConvGRUClassifier":
        model_cfg = getattr(config, "model", config)
        if isinstance(model_cfg, dict):
            cfg_get = lambda k, default: model_cfg.get(k, default)
        else:
            cfg_get = lambda k, default: getattr(model_cfg, k, default)

        in_dim = cfg_get("convgru_in_dim", cfg_get("input_dim", 128))
        if in_dim > 128:
            in_dim = 64

        hid_dim = cfg_get("convgru_hidden_dim", cfg_get("hidden_dim", 64))
        num_layers = cfg_get("convgru_num_layers", cfg_get("num_layers", 2))
        num_classes = cfg_get("num_classes", 2)
        spatial_in = cfg_get("cnn_neck_channels", (64, 128, 256))
        adapter_drop = cfg_get("adapter_dropout", 0.25)
        dropout = cfg_get("dropout", 0.35)
        supervision = cfg_get("supervision_mode", "attention_pooling")

        return cls(
            input_dim=in_dim,
            hidden_dim=hid_dim,
            num_layers=num_layers,
            kernel_size=3,
            num_classes=num_classes,
            spatial_in_channels=tuple(spatial_in),
            neck_dropout=adapter_drop,
            convgru_dropout=0.1,
            dropout=dropout,
            supervision_mode=supervision
        )

    def forward(
            self,
            features: Union[Tuple[torch.Tensor, ...], List[torch.Tensor], torch.Tensor],
            seq_lens: Optional[torch.Tensor] = None,
            hidden_state: Optional[List[torch.Tensor]] = None,
            return_sequence: bool = False,
            return_weights: bool = False,
            return_state: bool = False,
            *args: torch.Tensor
    ) -> Union[torch.Tensor, Tuple[Any, ...]]:
        if isinstance(seq_lens, torch.Tensor) and seq_lens.dim() >= 4:
            extra = [seq_lens]
            if isinstance(hidden_state, torch.Tensor) and hidden_state.dim() >= 4:
                extra.append(hidden_state)
                hidden_state = None
            if len(args) > 0:
                extra.extend(args)
            features = (features, *extra)
            seq_lens = None
        elif len(args) > 0 and isinstance(features, torch.Tensor):
            features = (features, *args)

        fused = self.spatial_neck(features, seq_lens=seq_lens)

        is_single_frame = (fused.dim() == 4)
        if is_single_frame:
            fused = fused.unsqueeze(1)

        st_out, last_states = self.convgru(
            fused,
            hidden_state=hidden_state,
            seq_lens=seq_lens
        )

        if return_weights:
            t_features, spatial_weights = self.spatial_pooling(st_out, seq_lens=seq_lens, return_weights=True)
        else:
            t_features = self.spatial_pooling(st_out, seq_lens=seq_lens, return_weights=False)
            spatial_weights = None

        is_seq_mode = return_sequence or (self.supervision_mode in ("sequence", "frame"))
        if is_seq_mode:
            logits = self.fc_out(t_features)
            if seq_lens is not None and not is_single_frame:
                b, t, _ = logits.shape
                time_indices = torch.arange(t, device=logits.device).unsqueeze(0)
                lens_expanded = seq_lens.to(logits.device).unsqueeze(1)
                mask = (time_indices < lens_expanded).unsqueeze(-1)
                logits = logits * mask.to(logits.dtype)
            temporal_weights = None
            if is_single_frame:
                logits = logits.squeeze(1)
        else:
            pooled, temporal_weights = self.temporal_pooling(t_features, seq_lens=seq_lens)
            logits = self.fc_out(pooled)

        outputs = [logits]
        if return_weights:
            outputs.append({
                "spatial": spatial_weights,
                "temporal": temporal_weights
            })
        if return_state:
            outputs.append(last_states)

        return tuple(outputs) if len(outputs) > 1 else logits


print("[✓] Đã định nghĩa lớp ConvGRUClassifier thành công!")


[✓] Đã định nghĩa lớp ConvGRUClassifier thành công!


## 📹 Section 5: Tăng cường Dữ liệu Video & Pipeline Nạp Video Thô
1. `VideoAugmenter`: Bộ tăng cường chuỗi khung hình từ `src/augment.py`. Sử dụng **CHUNG 1 RANDOM SEED** cho toàn bộ các frame trong 1 clip video để bảo toàn tính nhất quán thời gian (Temporal Consistency).
2. `RawVideoFramesDataset`: Giải mã video trực tiếp qua OpenCV, lấy mẫu đều đặn 0.2s, tạo tensor `uint8` siêu nhẹ trên CPU.
3. `collate_video_frames`: Ghép nối các chuỗi video có độ dài động và tạo mask đệm an toàn.


In [7]:
# 1. Bộ tăng cường chuỗi video nhất quán thời gian (VideoAugmenter từ src/augment.py)
def _make_gauss_noise(var_limit, p):
    params = inspect.signature(A.GaussNoise.__init__).parameters
    if "var_limit" in params:
        return A.GaussNoise(var_limit=var_limit, p=p)
    if "std_range" in params:
        var_min, var_max = var_limit
        std_min = max(0.0, min(1.0, (var_min ** 0.5) / 255.0))
        std_max = max(0.0, min(1.0, (var_max ** 0.5) / 255.0))
        return A.GaussNoise(std_range=(std_min, std_max), p=p)
    return A.GaussNoise(p=p)


def _make_shift_scale_rotate(shift_limit, scale_limit, rotate_limit, p, fill_color=(114, 114, 114)):
    if hasattr(A, "Affine"):
        scale = (1.0 - scale_limit, 1.0 + scale_limit) if isinstance(scale_limit, (int, float)) else (
            1.0 + scale_limit[0], 1.0 + scale_limit[1])
        translate = (-shift_limit, shift_limit) if isinstance(shift_limit, (int, float)) else shift_limit
        rotate = (-rotate_limit, rotate_limit) if isinstance(rotate_limit, (int, float)) else rotate_limit
        return A.Affine(scale=scale, translate_percent=translate, rotate=rotate, border_mode=cv2.BORDER_CONSTANT,
                        fill=fill_color, p=p)
    params = inspect.signature(A.ShiftScaleRotate.__init__).parameters
    kwargs = dict(shift_limit=shift_limit, scale_limit=scale_limit, rotate_limit=rotate_limit,
                  border_mode=cv2.BORDER_CONSTANT, p=p)
    kwargs["value" if "value" in params else "fill"] = fill_color
    return A.ShiftScaleRotate(**kwargs)


class VideoAugmenter:
    """
    Pipeline tăng cường dữ liệu cho chuỗi khung hình video.
    Bảo toàn tính nhất quán thời gian bằng cách áp dụng chung 1 random seed cho tất cả các frames trong cùng một clip.
    """

    def __init__(self, cfg: Optional[dict] = None):
        default_cfg = {
            "horizontalFlip": 0.5,
            "shiftScaleRotate": (0.08, 0.10, 15, 0.8),
            "randomBrightnessContrast": 0.4,
            "hueSaturationValue": (15, 20, 15, 0.3),
            "gaussNoise": (5.0, 25.0, 0.25),
            "blur": (5, 0.15),
            "p": 0.5
        }
        self.cfg = cfg or default_cfg
        shift_limit, scale_limit, rotate_limit, ssr_p = self.cfg["shiftScaleRotate"]
        hue_shift, sat_shift, val_shift, hsv_p = self.cfg["hueSaturationValue"]
        var_min, var_max, gn_p = self.cfg["gaussNoise"]
        blur_limit, blur_p = self.cfg["blur"]

        self.transform = A.Compose(
            [
                A.HorizontalFlip(p=self.cfg["horizontalFlip"]),
                _make_shift_scale_rotate(shift_limit, scale_limit, rotate_limit, ssr_p),
                A.RandomBrightnessContrast(p=self.cfg["randomBrightnessContrast"]),
                A.HueSaturationValue(hue_shift_limit=hue_shift, sat_shift_limit=sat_shift, val_shift_limit=val_shift,
                                     p=hsv_p),
                _make_gauss_noise((var_min, var_max), p=gn_p),
                A.Blur(blur_limit=blur_limit, p=blur_p),
            ],
            p=self.cfg["p"]
        )

    def set_seed(self, seed: int):
        random.seed(seed)
        np.random.seed(seed)
        if hasattr(self.transform, "set_random_seed"):
            self.transform.set_random_seed(seed)

    def apply_sequence(self, frames: List[np.ndarray], seed: Optional[int] = None) -> List[np.ndarray]:
        """Áp dụng cùng một seed biến đổi cho tất cả các frames của 1 video clip."""
        if seed is None:
            seed = random.randint(0, 2 ** 31 - 1)
        aug_frames = []
        for frame in frames:
            self.set_seed(seed)
            try:
                res = self.transform(image=frame)
                aug_frames.append(res["image"])
            except Exception:
                aug_frames.append(frame)
        return aug_frames


print("[✓] Đã định nghĩa lớp VideoAugmenter từ src/augment.py thành công!")


[✓] Đã định nghĩa lớp VideoAugmenter từ src/augment.py thành công!


In [8]:
# 2. Pipeline tiền xử lý và nạp dữ liệu Video Thô
def letterbox(image: np.ndarray, new_size: int = 640, color: Tuple[int, int, int] = (114, 114, 114)) -> np.ndarray:
    """Resize ảnh giữ nguyên tỉ lệ khung hình (aspect ratio) với padding đồng màu."""
    h, w = image.shape[:2]
    scale = min(new_size / h, new_size / w)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    canvas = np.full((new_size, new_size, 3), color, dtype=image.dtype)
    pad_left = (new_size - new_w) // 2
    pad_top = (new_size - new_h) // 2
    canvas[pad_top: pad_top + new_h, pad_left: pad_left + new_w] = resized
    return canvas


@dataclass
class RawVideoSample:
    path: Path
    video_id: str
    split: str
    label: int
    label_name: str
    source_dataset: str = "custom"
    subject_id: Optional[str] = None


class RawVideoFramesDataset(Dataset):
    """
    Dataset đọc trực tiếp video thô (.mp4, .avi, .mkv) bằng OpenCV.
    Chạy 100% trên CPU, trả về Tensor uint8 tiết kiệm RAM và bảo đảm an toàn đa tiến trình.
    """

    def __init__(
            self,
            dataset_dir: Union[str, Path],
            manifest_file: Optional[str] = None,
            split: str = "train",
            sample_interval: float = 0.2,
            seq_len: Optional[int] = None,
            img_size: int = 640,
            video_exts: Tuple[str, ...] = (".avi", ".mp4", ".mkv", ".mov"),
            min_frames: int = 10,
            augmenter: Optional[Any] = None,
            window_sampling: str = "random"
    ):
        self.dataset_dir = Path(dataset_dir)
        self.manifest_file = manifest_file
        self.split = split.lower().strip()
        self.sample_interval = float(sample_interval)
        self.seq_len = seq_len
        self.img_size = int(img_size)
        self.video_exts = tuple(ext.lower() for ext in video_exts)
        self.min_frames = max(1, int(min_frames))
        self.augmenter = augmenter
        self.window_sampling = window_sampling
        self.samples = self._load_samples()

    def _load_samples(self) -> List[RawVideoSample]:
        samples = []
        manifest_path = None
        if self.manifest_file:
            cand = self.dataset_dir / self.manifest_file
            if cand.exists():
                manifest_path = cand

        if manifest_path is None:
            for cand_name in ("dataset_manifest.csv", "dataset_merged_split.csv"):
                cand = self.dataset_dir / cand_name
                if cand.exists():
                    manifest_path = cand
                    break

        if manifest_path and manifest_path.exists():
            print(f"[*] [{self.split.upper()}] Đang nạp danh sách mẫu từ manifest: {manifest_path.name}")
            with open(manifest_path, mode="r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    r_split = row.get("split", "").lower().strip()
                    if r_split != self.split:
                        continue
                    v_path = self.dataset_dir / row.get("processed_path", row.get("path", ""))
                    if not v_path.exists():
                        # Thử tìm trực tiếp theo tên file
                        v_path = self.dataset_dir / self.split / row.get("processed_path",
                                                                         Path(row.get("path", "")).name)
                        if not v_path.exists():
                            continue
                    label = int(row.get("binary_label", 0))
                    label_name = row.get("label_name", "drowsy" if label == 1 else "alert")
                    samples.append(RawVideoSample(
                        path=v_path,
                        video_id=row.get("video_id", v_path.stem),
                        split=self.split,
                        label=label,
                        label_name=label_name,
                        source_dataset=row.get("source_dataset", "custom"),
                        subject_id=row.get("subject_id")
                    ))
        else:
            print(f"[*] [{self.split.upper()}] Quét trực tiếp từ cấu trúc thư mục con: {self.dataset_dir / self.split}")
            split_dir = self.dataset_dir / self.split
            label_map = [(0, "0_alert"), (1, "1_drowsy")]
            for label, folder_name in label_map:
                f_dir = split_dir / folder_name
                if not f_dir.exists():
                    continue
                for file_p in f_dir.iterdir():
                    if file_p.suffix.lower() in self.video_exts:
                        samples.append(RawVideoSample(
                            path=file_p,
                            video_id=file_p.stem,
                            split=self.split,
                            label=label,
                            label_name="drowsy" if label == 1 else "alert"
                        ))
        return samples

    def _sample_video_frames(self, video_path: Path) -> Tuple[List[np.ndarray], float, float]:
        cap = cv2.VideoCapture(str(video_path))
        if not cap.isOpened():
            print(f"Lỗi đọc video (bỏ qua): {video_path}")
            return [], 0.0, 0.0

        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        duration_s = total_frames / fps if fps > 0 else 0.0

        step = max(1, int(round(fps * self.sample_interval)))
        frames_rgb = []
        frame_idx = 0

        while True:
            ret, frame_bgr = cap.read()
            if not ret or frame_bgr is None:
                break
            if frame_idx % step == 0:
                frame_lb = letterbox(frame_bgr, new_size=self.img_size)
                frame_rgb = cv2.cvtColor(frame_lb, cv2.COLOR_BGR2RGB)
                frames_rgb.append(frame_rgb)
            frame_idx += 1

        cap.release()
        return frames_rgb, float(fps), float(duration_s)

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Tuple[Optional[torch.Tensor], torch.Tensor, torch.Tensor, Dict[str, Any]]:
        sample = self.samples[idx]
        frames_rgb, fps, duration_s = self._sample_video_frames(sample.path)
        orig_sampled_len = len(frames_rgb)

        if orig_sampled_len < self.min_frames:
            meta = {"video_id": sample.video_id, "status": "corrupted"}
            return None, torch.tensor(sample.label), torch.tensor(0), meta

        if self.seq_len is not None and len(frames_rgb) > self.seq_len:
            if self.split == "train" and self.window_sampling == "random":
                start_idx = random.randint(0, len(frames_rgb) - self.seq_len)
            else:
                start_idx = (len(frames_rgb) - self.seq_len) // 2
            frames_rgb = frames_rgb[start_idx: start_idx + self.seq_len]

        # Áp dụng Tăng cường chuỗi video đồng bộ thời gian cho tập train
        if self.augmenter is not None and self.split == "train":
            aug_seed = random.randint(0, 2 ** 31 - 1)
            frames_rgb = self.augmenter.apply_sequence(frames_rgb, seed=aug_seed)

        # Chuyển đổi sang Tensor uint8 [T, 3, H, W] trên CPU
        batch_np = np.stack(frames_rgb, axis=0).transpose(0, 3, 1, 2)
        frames_t = torch.from_numpy(batch_np).to(dtype=torch.uint8)
        label_t = torch.tensor(sample.label, dtype=torch.long)
        seq_len_t = torch.tensor(len(frames_rgb), dtype=torch.long)

        meta = {
            "video_id": sample.video_id,
            "video_path": str(sample.path),
            "split": sample.split,
            "label": sample.label,
            "label_name": sample.label_name,
            "fps": fps,
            "sample_interval": self.sample_interval
        }
        return frames_t, label_t, seq_len_t, meta


def collate_video_frames(batch):
    """Gom nhóm các video clip có độ dài thay đổi theo cơ chế Dynamic Padding."""
    valid_samples = [item for item in batch if item[0] is not None]
    if not valid_samples:
        return torch.empty(0), torch.empty(0), torch.empty(0), []

    frames_list = [item[0] for item in valid_samples]
    labels_list = [item[1] for item in valid_samples]
    seq_lens_list = [item[2] for item in valid_samples]
    metas_list = [item[3] for item in valid_samples]

    max_len = max(f.shape[0] for f in frames_list)
    B = len(frames_list)
    _, C, H, W = frames_list[0].shape

    padded_frames = torch.zeros(B, max_len, C, H, W, dtype=torch.uint8)
    for i, f in enumerate(frames_list):
        T = f.shape[0]
        padded_frames[i, :T] = f

    labels_t = torch.stack(labels_list)
    seq_lens_t = torch.stack(seq_lens_list)
    return padded_frames, labels_t, seq_lens_t, metas_list


def build_raw_video_dataloaders(
        dataset_dir: Union[str, Path],
        manifest_file: Optional[str] = None,
        sample_interval: float = 0.2,
        seq_len: Optional[int] = None,
        batch_size: int = 4,
        val_batch_size: Optional[int] = None,
        num_workers: int = 2,
        val_num_workers: Optional[int] = None,
        pin_memory: bool = False,
        shuffle_train: bool = True,
        use_augmentation: bool = True,
        min_frames: int = 10,
        img_size: int = 640
) -> Tuple[DataLoader, DataLoader]:
    val_b_size = val_batch_size if val_batch_size is not None else batch_size
    val_n_workers = val_num_workers if val_num_workers is not None else num_workers

    augmenter = VideoAugmenter() if use_augmentation else None

    train_dataset = RawVideoFramesDataset(
        dataset_dir=dataset_dir,
        manifest_file=manifest_file,
        split="train",
        sample_interval=sample_interval,
        seq_len=seq_len,
        img_size=img_size,
        min_frames=min_frames,
        augmenter=augmenter
    )

    val_dataset = RawVideoFramesDataset(
        dataset_dir=dataset_dir,
        manifest_file=manifest_file,
        split="val",
        sample_interval=sample_interval,
        seq_len=seq_len,
        img_size=img_size,
        min_frames=min_frames,
        augmenter=None
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle_train,
        num_workers=num_workers,
        pin_memory=pin_memory,
        collate_fn=collate_video_frames,
        drop_last=False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=val_b_size,
        shuffle=False,
        num_workers=val_n_workers,
        pin_memory=pin_memory,
        collate_fn=collate_video_frames,
        drop_last=False
    )

    return train_loader, val_loader


print("[✓] Đã định nghĩa DataLoader và RawVideoFramesDataset thành công!")


[✓] Đã định nghĩa DataLoader và RawVideoFramesDataset thành công!


## 🛠️ Section 6: Hạ tầng Huấn luyện & Cơ chế Tiếp tục Huấn luyện (Trainer)
Định nghĩa đầy đủ bộ điều phối `Trainer`:
- Bắt lỗi CUDA Out-Of-Memory (OOM) an toàn cho cả Train và Val loops.
- Tích lũy gradient (Gradient Accumulation) chuẩn xác ở biên epoch.
- Bộ điều chỉnh Mixed Precision FP16 Resumable GradScaler.
- Cơ chế Atomic Save chống hỏng file checkpoint (`best.pt`, `last.pt`, `epoch_N.pt`).
- Tự động ghi nhật ký lịch sử huấn luyện ra `training_history.csv`.


In [9]:
def calculate_metrics(preds: np.ndarray, targets: np.ndarray) -> Dict[str, float]:
    """Tính toán Accuracy, Recall, F1-Score (macro) an toàn từ mảng NumPy."""
    if len(preds) == 0 or len(targets) == 0:
        return {"acc": 0.0, "recall": 0.0, "f1": 0.0}
    acc = accuracy_score(targets, preds)
    recall = recall_score(targets, preds, average='macro', zero_division=0)
    f1 = f1_score(targets, preds, average='macro', zero_division=0)
    return {"acc": float(acc), "recall": float(recall), "f1": float(f1)}


class EarlyStopping:
    """Bộ giám sát dừng sớm Early Stopping dựa trên metric chỉ định."""

    def __init__(self, patience: int = 10, min_delta: float = 1e-4, mode: str = "max", monitor: str = "val_f1"):
        self.patience = max(1, patience)
        self.min_delta = float(min_delta)
        self.mode = mode.lower().strip()
        self.monitor = monitor
        self.best_score = -float("inf") if self.mode == "max" else float("inf")
        self.counter = 0
        self.best_epoch = 0
        self.early_stop = False

    def step(self, current_val: float, epoch: int) -> bool:
        if self.mode == "max":
            improved = current_val > (self.best_score + self.min_delta)
        else:
            improved = current_val < (self.best_score - self.min_delta)

        if improved:
            self.best_score = current_val
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop


class Trainer:
    """Pipeline Huấn luyện và Kiểm định Chuyên nghiệp cho ConvGRUClassifier trên Kaggle."""

    def __init__(self, config: TrainConfig):
        self.cfg = config
        self.device = torch.device(self.cfg.device if torch.cuda.is_available() else "cpu")
        print(f"[*] Đang sử dụng thiết bị: {self.device}")

        # Khởi tạo DataLoader
        print("[*] Đang khởi tạo DataLoader từ dataset video thô...")
        val_batch_size = getattr(self.cfg, "val_batch_size", self.cfg.batch_size)
        val_num_workers = getattr(self.cfg, "val_num_workers", self.cfg.num_workers)
        min_frames = getattr(self.cfg, "min_frames", 10)
        img_size = getattr(self.cfg, "image_size", (640, 640))[0]

        self.train_loader, self.val_loader = build_raw_video_dataloaders(
            dataset_dir=self.cfg.dataset_dir,
            manifest_file=self.cfg.manifest_file,
            sample_interval=self.cfg.sample_interval,
            seq_len=self.cfg.seq_len,
            batch_size=self.cfg.batch_size,
            val_batch_size=val_batch_size,
            num_workers=self.cfg.num_workers,
            val_num_workers=val_num_workers,
            pin_memory=self.cfg.pin_memory,
            shuffle_train=self.cfg.shuffle,
            use_augmentation=self.cfg.use_augmentation,
            min_frames=min_frames,
            img_size=img_size
        )

        train_count = len(self.train_loader.dataset)
        val_count = len(self.val_loader.dataset)
        print(f"[✓] Tập Train: {train_count} video | Tập Val: {val_count} video")
        if train_count == 0:
            print(
                f"[!] CẢNH BÁO: Tập huấn luyện không tìm thấy video nào trong '{self.cfg.dataset_dir}'. Hãy kiểm tra lại đường dẫn!")

        # Khởi tạo mô hình trích xuất đặc trưng BackboneNeck trên GPU
        print("[*] Đang nạp mô hình trích xuất Backbone PAFPN...")
        self.extractor = ChunkedBackboneNeckExtractor(
            checkpoint_path=self.cfg.backbone_neck_checkpoint,
            device=self.cfg.device,
            chunk_size=self.cfg.chunk_size,
            use_fp16=self.cfg.amp,
            img_size=img_size
        ).to(self.device)

        # Khởi tạo mô hình ConvGRUClassifier chính
        print("[*] Đang nạp mô hình ConvGRUClassifier...")
        self.model = ConvGRUClassifier.from_config(self.cfg).to(self.device)
        total_params = sum(p.numel() for p in self.model.parameters())
        print(f"[✓] Tổng số tham số ConvGRUClassifier: {total_params:,}")

        # Cấu hình hàm mất mát (Loss)
        num_classes = getattr(self.cfg, "num_classes", 2)
        if self.cfg.loss_type.lower() == "bce" and num_classes == 1:
            pos_weight = torch.tensor([self.cfg.pos_weight], device=self.device) if self.cfg.pos_weight else None
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=self.cfg.bce_reduction)
            self.is_binary_bce = True
        else:
            self.criterion = nn.CrossEntropyLoss()
            self.is_binary_bce = False

        # Optimizer & Scheduler
        if self.cfg.optimizer.lower() == "sgd":
            self.optimizer = optim.SGD(self.model.parameters(), lr=self.cfg.lr0, momentum=self.cfg.momentum,
                                       weight_decay=self.cfg.weight_decay)
        else:
            self.optimizer = optim.AdamW(self.model.parameters(), lr=self.cfg.lr0, betas=self.cfg.betas,
                                         weight_decay=self.cfg.weight_decay)

        if self.cfg.use_scheduler:
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer,
                T_max=self.cfg.epochs,
                eta_min=self.cfg.lr0 * self.cfg.lr_min_factor
            )
        else:
            self.scheduler = None

        # Thư mục Checkpoint & TensorBoard
        self.ckpt_dir = Path(self.cfg.checkpoint_dir)
        self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        tb_dir = Path(self.cfg.tb_log_dir) / self.cfg.experiment_name
        tb_dir.mkdir(parents=True, exist_ok=True)
        self.writer = SummaryWriter(log_dir=str(tb_dir))

        # AMP GradScaler
        self.scaler = torch.amp.GradScaler("cuda") if (self.cfg.amp and self.device.type == "cuda") else None

        # Early Stopping
        if getattr(self.cfg, "early_stopping", False):
            self.early_stopper = EarlyStopping(
                patience=getattr(self.cfg, "patience", 10),
                min_delta=getattr(self.cfg, "min_delta", 1e-4),
                mode=getattr(self.cfg, "monitor_mode", "max"),
                monitor=getattr(self.cfg, "monitor_metric", "val_f1")
            )
        else:
            self.early_stopper = None

        # File training_history.csv
        self.history_csv_path = Path(self.cfg.history_csv_path)
        self._init_history_csv()

        self.global_step = 0
        self.best_val_f1 = 0.0
        self.start_epoch = 1

    def _init_history_csv(self) -> None:
        self.history_csv_path.parent.mkdir(parents=True, exist_ok=True)
        if not self.history_csv_path.exists():
            with open(self.history_csv_path, mode="w", newline="", encoding="utf-8") as f:
                writer = csv.writer(f)
                writer.writerow([
                    "epoch", "train_loss", "train_acc", "train_recall", "train_f1",
                    "val_loss", "val_acc", "val_recall", "val_f1", "lr", "time_sec"
                ])

    def _log_history_csv(self, epoch: int, train_m: Dict[str, float], val_m: Optional[Dict[str, float]],
                         elapsed_sec: float) -> None:
        lr = self.optimizer.param_groups[0]['lr']
        val_loss = f"{val_m['loss']:.4f}" if val_m else ""
        val_acc = f"{val_m['acc']:.4f}" if val_m else ""
        val_rec = f"{val_m['recall']:.4f}" if val_m else ""
        val_f1 = f"{val_m['f1']:.4f}" if val_m else ""

        with open(self.history_csv_path, mode="a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                epoch,
                f"{train_m['loss']:.4f}",
                f"{train_m['acc']:.4f}",
                f"{train_m['recall']:.4f}",
                f"{train_m['f1']:.4f}",
                val_loss,
                val_acc,
                val_rec,
                val_f1,
                f"{lr:.6e}",
                f"{elapsed_sec:.2f}"
            ])

    def compute_loss_and_preds(self, logits: torch.Tensor, labels: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        if self.is_binary_bce:
            logits_1d = logits.view(-1)
            loss = self.criterion(logits_1d, labels.float().view(-1))
            preds = (torch.sigmoid(logits_1d) >= 0.5).long()
        else:
            loss = self.criterion(logits, labels.long())
            preds = torch.argmax(logits, dim=1)
        return loss, preds

    def train_epoch(self, epoch: int) -> Dict[str, float]:
        self.model.train()
        total_loss = 0.0
        all_preds = []
        all_targets = []
        valid_batches = 0

        accum_steps = max(1, getattr(self.cfg, "gradient_accumulation_steps", 1))
        accum_count = 0
        self.optimizer.zero_grad()

        num_batches = len(self.train_loader)
        pbar = tqdm(self.train_loader, desc=f"Train Epoch {epoch:02d}/{self.cfg.epochs}", disable=not self.cfg.use_tqdm)

        for batch_idx, batch in enumerate(pbar):
            frames_t, labels, seq_lens, metas = batch
            if frames_t.size(0) == 0:
                continue

            frames_t = frames_t.to(self.device)
            labels = labels.to(self.device)
            seq_lens = seq_lens.to(self.device)

            try:
                autocast_enabled = (self.scaler is not None and self.device.type == "cuda")
                with torch.amp.autocast(device_type=self.device.type, enabled=autocast_enabled):
                    p3, p4, p5 = self.extractor(frames_t)
                    logits = self.model((p3, p4, p5), seq_lens=seq_lens)
                    loss, preds = self.compute_loss_and_preds(logits, labels)
                    loss_scaled = loss / accum_steps

                accum_count += 1
                if self.scaler is not None:
                    self.scaler.scale(loss_scaled).backward()
                else:
                    loss_scaled.backward()

                is_boundary = (accum_count == accum_steps) or ((batch_idx + 1) == num_batches)
                if is_boundary:
                    if self.scaler is not None:
                        self.scaler.unscale_(self.optimizer)
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                        self.scaler.step(self.optimizer)
                        self.scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.grad_clip_norm)
                        self.optimizer.step()
                    self.optimizer.zero_grad()
                    accum_count = 0

                unscaled_loss = loss.item()
                total_loss += unscaled_loss
                valid_batches += 1

                preds_np = preds.detach().cpu().numpy()
                targets_np = labels.detach().cpu().numpy()
                all_preds.extend(preds_np)
                all_targets.extend(targets_np)
                self.global_step += 1

                if self.cfg.enable_step_logging and self.global_step % self.cfg.log_step_interval == 0:
                    step_metrics = calculate_metrics(preds_np, targets_np)
                    self.writer.add_scalar("Train_Step/Loss", unscaled_loss, self.global_step)
                    self.writer.add_scalar("Train_Step/Accuracy", step_metrics["acc"], self.global_step)
                    self.writer.add_scalar("Train_Step/F1", step_metrics["f1"], self.global_step)

                pbar.set_postfix({"loss": f"{unscaled_loss:.4f}"})

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"[!] CUDA Out of Memory tại batch {batch_idx}. Đang dọn cache GPU...")
                    self.optimizer.zero_grad()
                    accum_count = 0
                    try:
                        del frames_t, labels, seq_lens, metas
                    except Exception:
                        pass
                    del e
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                else:
                    raise e

        avg_loss = total_loss / max(valid_batches, 1)
        epoch_metrics = calculate_metrics(np.array(all_preds), np.array(all_targets))
        epoch_metrics["loss"] = avg_loss
        return epoch_metrics

    @torch.no_grad()
    def validate_epoch(self, epoch: int) -> Dict[str, float]:
        self.model.eval()
        total_loss = 0.0
        all_preds = []
        all_targets = []
        valid_batches = 0

        pbar = tqdm(self.val_loader, desc=f"Val Epoch   {epoch:02d}/{self.cfg.epochs}", disable=not self.cfg.use_tqdm)
        for batch_idx, batch in enumerate(pbar):
            frames_t, labels, seq_lens, metas = batch
            if frames_t.size(0) == 0:
                continue

            frames_t = frames_t.to(self.device)
            labels = labels.to(self.device)
            seq_lens = seq_lens.to(self.device)

            try:
                autocast_enabled = (self.cfg.amp and self.device.type == "cuda")
                with torch.amp.autocast(device_type=self.device.type, enabled=autocast_enabled):
                    p3, p4, p5 = self.extractor(frames_t)
                    logits = self.model((p3, p4, p5), seq_lens=seq_lens)
                    loss, preds = self.compute_loss_and_preds(logits, labels)

                total_loss += loss.item()
                valid_batches += 1
                all_preds.extend(preds.cpu().numpy())
                all_targets.extend(labels.cpu().numpy())

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"[!] CUDA Out of Memory tại Val batch {batch_idx}. Bỏ qua batch...")
                    try:
                        del frames_t, labels, seq_lens, metas
                    except Exception:
                        pass
                    del e
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                else:
                    raise e

        avg_loss = total_loss / max(valid_batches, 1)
        epoch_metrics = calculate_metrics(np.array(all_preds), np.array(all_targets))
        epoch_metrics["loss"] = avg_loss
        return epoch_metrics

    def _resolve_checkpoint_path(self, checkpoint_path: str) -> Path:
        alias = str(checkpoint_path).strip().lower()
        if alias in ("last", "last.pt"):
            return self.ckpt_dir / "last.pt"
        elif alias in ("best", "best.pt"):
            return self.ckpt_dir / "best.pt"
        target = Path(checkpoint_path)
        if not target.is_absolute():
            cand = self.ckpt_dir / target
            if cand.exists():
                return cand
        return target

    def load_checkpoint(self, checkpoint_path: str) -> None:
        target_path = self._resolve_checkpoint_path(checkpoint_path)
        print(f"[*] Đang khôi phục từ checkpoint: {target_path}")
        if not target_path.exists():
            raise FileNotFoundError(f"Không tìm thấy checkpoint tại: {target_path}")

        ckpt = torch.load(target_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        self.start_epoch = ckpt.get("epoch", 0) + 1
        self.best_val_f1 = ckpt.get("best_val_f1", 0.0)
        self.global_step = ckpt.get("global_step", (self.start_epoch - 1) * len(self.train_loader))

        if self.scheduler is not None and "scheduler_state_dict" in ckpt:
            self.scheduler.load_state_dict(ckpt["scheduler_state_dict"])
        if self.scaler is not None and "scaler_state_dict" in ckpt:
            self.scaler.load_state_dict(ckpt["scaler_state_dict"])
            print("[✓] Đã khôi phục thành công trạng thái GradScaler (AMP FP16).")

        print(f"[✓] Tiếp tục huấn luyện từ epoch {self.start_epoch} (Best F1 trước đó: {self.best_val_f1:.4f})")

    def _atomic_save(self, state: Dict[str, Any], target_path: Path) -> None:
        tmp_path = target_path.with_suffix(f"{target_path.suffix}.tmp")
        torch.save(state, tmp_path)
        if target_path.exists():
            target_path.unlink()
        tmp_path.rename(target_path)

    def save_checkpoint(self, epoch: int, is_best: bool = False, val_f1: Optional[float] = None) -> None:
        state = {
            "epoch": epoch,
            "global_step": self.global_step,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "best_val_f1": self.best_val_f1,
            "val_f1": val_f1,
            "config": self.cfg.to_dict()
        }
        if self.scheduler is not None:
            state["scheduler_state_dict"] = self.scheduler.state_dict()
        if self.scaler is not None:
            state["scaler_state_dict"] = self.scaler.state_dict()

        # 1. Luôn lưu last.pt
        self._atomic_save(state, self.ckpt_dir / "last.pt")

        # 2. Lưu best.pt nếu F1 cao nhất
        if is_best:
            self._atomic_save(state, self.ckpt_dir / "best.pt")
            print(f"[★] Đã lưu mô hình tốt nhất (best.pt) tại epoch {epoch} với F1: {self.best_val_f1:.4f}")

        # 3. Lưu định kỳ theo epoch
        save_all = getattr(self.cfg, "save_all_epochs", False)
        save_interval = getattr(self.cfg, "save_ckpt_interval_epochs", 5)
        if save_all or (save_interval > 0 and epoch % save_interval == 0):
            self._atomic_save(state, self.ckpt_dir / f"epoch_{epoch}.pt")

        # 4. Tỉa bớt checkpoint cũ nếu cần
        keep_last = getattr(self.cfg, "ckpt_keep_last", None)
        if keep_last and keep_last > 0:
            epoch_files = sorted(
                self.ckpt_dir.glob("epoch_*.pt"),
                key=lambda p: int(p.stem.split("_")[1]) if p.stem.split("_")[1].isdigit() else 0
            )
            if len(epoch_files) > keep_last:
                for old_f in epoch_files[:-keep_last]:
                    try:
                        old_f.unlink()
                    except Exception:
                        pass

    def train(self) -> None:
        print(f"[*] Bắt đầu vòng lặp huấn luyện từ epoch {self.start_epoch} đến {self.cfg.epochs}...")
        for epoch in range(self.start_epoch, self.cfg.epochs + 1):
            t_epoch_start = time.time()
            train_metrics = self.train_epoch(epoch)

            # Ghi log TensorBoard Train
            self.writer.add_scalar("Train_Epoch/Loss", train_metrics["loss"], epoch)
            self.writer.add_scalar("Train_Epoch/Accuracy", train_metrics["acc"], epoch)
            self.writer.add_scalar("Train_Epoch/Recall", train_metrics["recall"], epoch)
            self.writer.add_scalar("Train_Epoch/F1", train_metrics["f1"], epoch)
            self.writer.add_scalar("Train_Epoch/LearningRate", self.optimizer.param_groups[0]['lr'], epoch)

            # Đánh giá tập Validation
            run_val = (len(self.val_loader.dataset) > 0) and (
                    (epoch % self.cfg.val_interval_epochs == 0) or (epoch == self.cfg.epochs)
            )

            is_best = False
            val_metrics = None
            if run_val:
                val_metrics = self.validate_epoch(epoch)
                self.writer.add_scalar("Val_Epoch/Loss", val_metrics["loss"], epoch)
                self.writer.add_scalar("Val_Epoch/Accuracy", val_metrics["acc"], epoch)
                self.writer.add_scalar("Val_Epoch/Recall", val_metrics["recall"], epoch)
                self.writer.add_scalar("Val_Epoch/F1", val_metrics["f1"], epoch)

                print(
                    f"Epoch {epoch:02d}/{self.cfg.epochs} | Train Loss: {train_metrics['loss']:.4f} | Train F1: {train_metrics['f1']:.4f} | Train Acc: {train_metrics['acc']:.4f} | Val Recall: {train_metrics["recall"]} |")
                print(
                    f"Val Loss: {val_metrics['loss']:.4f} | Val F1: {val_metrics['f1']:.4f} | Val Acc: {val_metrics['acc']:.4f} | Val Recall: {val_metrics["recall"]}")

                if val_metrics["f1"] > self.best_val_f1:
                    self.best_val_f1 = val_metrics["f1"]
                    is_best = True
            else:
                print(
                    f"Epoch {epoch:02d}/{self.cfg.epochs} | Train Loss: {train_metrics['loss']:.4f} | Train F1: {train_metrics['f1']:.4f} | (Val: Bỏ qua)")

            # Lưu checkpoint cuối epoch
            self.save_checkpoint(epoch=epoch, is_best=is_best, val_f1=val_metrics["f1"] if val_metrics else None)

            # Ghi nhận vào file training_history.csv
            elapsed_sec = time.time() - t_epoch_start
            self._log_history_csv(epoch, train_metrics, val_metrics, elapsed_sec)

            # Kiểm tra Early Stopping
            if run_val and self.early_stopper is not None and val_metrics is not None:
                monitor_key = self.early_stopper.monitor.replace("val_", "")
                target_score = val_metrics.get(monitor_key,
                                               val_metrics.get(self.early_stopper.monitor, val_metrics["f1"]))
                if self.early_stopper.step(target_score, epoch):
                    print(
                        f"[*] Early Stopping kích hoạt tại epoch {epoch}! (Kỷ nguyên tốt nhất: {self.early_stopper.best_epoch})")
                    break

            if self.scheduler is not None:
                self.scheduler.step()

            if self.cfg.empty_cache_interval > 0 and epoch % self.cfg.empty_cache_interval == 0:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        print("[✓] Đã hoàn thành quá trình huấn luyện!")
        self.writer.close()


print("[✓] Đã định nghĩa lớp Trainer chuyên nghiệp thành công!")


[✓] Đã định nghĩa lớp Trainer chuyên nghiệp thành công!


## 🚀 Section 7: Thực thi Huấn luyện Tương tác
Khởi tạo Trainer, xác nhận số lượng tham số mô hình đạt chuẩn 631,716 tham số và bắt đầu vòng lặp huấn luyện.


In [10]:
# 1. Khởi tạo đối tượng Trainer và kiểm tra cấu trúc mô hình
trainer = Trainer(cfg)

# Kiểm tra xác thực số lượng tham số mô hình ConvGRUClassifier
total_params = sum(p.numel() for p in trainer.model.parameters())
trainable_params = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)

print("=" * 60)
print(f"[*] Tổng số tham số ConvGRUClassifier : {total_params:,}")
print(f"[*] Số tham số huấn luyện (Trainable) : {trainable_params:,}")
print("=" * 60)

# Khôi phục từ checkpoint nếu cấu hình yêu cầu
if cfg.enable_resume and cfg.resume:
    trainer.load_checkpoint(cfg.resume)


[*] Đang sử dụng thiết bị: cuda
[*] Đang khởi tạo DataLoader từ dataset video thô...
[*] [TRAIN] Đang nạp danh sách mẫu từ manifest: dataset_merged_split.csv
[*] [VAL] Đang nạp danh sách mẫu từ manifest: dataset_merged_split.csv
[✓] Tập Train: 1615 video | Tập Val: 352 video
[*] Đang nạp mô hình trích xuất Backbone PAFPN...
[*] Nạp checkpoint Backbone từ: D:\Project\DATN\driver-guardian\ai\ObjectDetection_2p6M\checkpoints\231e35bb4061257f9bcb7cc5e3a0d0064e5e1beb6351a5e0adfc8d0c91ba4e11\da92589d1b67339be4491beaf147575e81a2699bbde06d654c05f852d7f0e9c9\landmark_train\best.pt
[✓] Đã nạp thành công 444 tensors trọng số vào BackboneNeck.
[*] Đang nạp mô hình ConvGRUClassifier...
[✓] Tổng số tham số ConvGRUClassifier: 631,716
[*] Tổng số tham số ConvGRUClassifier : 631,716
[*] Số tham số huấn luyện (Trainable) : 631,716
[*] Đang khôi phục từ checkpoint: D:\Project\DATN\driver-guardian\ai\LSTM\checkpoints\experiments\deepgru_raw_nmsfree\epoch_5.pt
[✓] Đã khôi phục thành công trạng thái GradSca

In [ ]:
# 2. Bắt đầu quá trình huấn luyện mô hình!
# Quá trình này sẽ hiển thị thanh tiến trình trực quan tqdm qua từng epoch
trainer.train()


[*] Bắt đầu vòng lặp huấn luyện từ epoch 6 đến 20...


Train Epoch 06/20:   0%|          | 0/808 [00:00<?, ?it/s]

In [ ]:
# 1. Vẽ đồ thị Loss, Accuracy, F1 và Learning Rate qua các epochs
history_file = Path(cfg.history_csv_path)

if history_file.exists():
    epochs_data = []
    train_loss, val_loss = [], []
    train_f1, val_f1 = [], []
    train_acc, val_acc = [], []
    lr_data = []

    with open(history_file, mode="r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            epochs_data.append(int(row["epoch"]))
            train_loss.append(float(row["train_loss"]))
            val_loss.append(float(row["val_loss"]) if row["val_loss"] else None)
            train_f1.append(float(row["train_f1"]))
            val_f1.append(float(row["val_f1"]) if row["val_f1"] else None)
            train_acc.append(float(row["train_acc"]))
            val_acc.append(float(row["val_acc"]) if row["val_acc"] else None)
            lr_data.append(float(row["lr"]))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss
    axes[0, 0].plot(epochs_data, train_loss, label="Train Loss", marker="o", color="#1f77b4")
    if any(v is not None for v in val_loss):
        axes[0, 0].plot(epochs_data, val_loss, label="Val Loss", marker="s", color="#ff7f0e")
    axes[0, 0].set_title("Hàm mất mát (Loss) qua các Epoch")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].grid(True, linestyle="--", alpha=0.6)
    axes[0, 0].legend()

    # F1 Score
    axes[0, 1].plot(epochs_data, train_f1, label="Train F1", marker="o", color="#2ca02c")
    if any(v is not None for v in val_f1):
        axes[0, 1].plot(epochs_data, val_f1, label="Val F1", marker="s", color="#d62728")
    axes[0, 1].set_title("F1-Score qua các Epoch")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("Macro F1")
    axes[0, 1].grid(True, linestyle="--", alpha=0.6)
    axes[0, 1].legend()

    # Accuracy
    axes[1, 0].plot(epochs_data, train_acc, label="Train Accuracy", marker="o", color="#9467bd")
    if any(v is not None for v in val_acc):
        axes[1, 0].plot(epochs_data, val_acc, label="Val Accuracy", marker="s", color="#8c564b")
    axes[1, 0].set_title("Độ chính xác (Accuracy) qua các Epoch")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Accuracy")
    axes[1, 0].grid(True, linestyle="--", alpha=0.6)
    axes[1, 0].legend()

    # Learning Rate
    axes[1, 1].plot(epochs_data, lr_data, label="Learning Rate", marker="^", color="#e377c2")
    axes[1, 1].set_title("Tốc độ học (Learning Rate Cosine Annealing)")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("LR")
    axes[1, 1].grid(True, linestyle="--", alpha=0.6)
    axes[1, 1].legend()

    plt.tight_layout()
    chart_path = Path("/kaggle/working/training_curves.png")
    plt.savefig(chart_path, dpi=300)
    plt.show()
    print(f"[✓] Đã xuất biểu đồ học tập lưu tại: {chart_path}")
else:
    print("[!] Không tìm thấy file training_history.csv để vẽ biểu đồ.")


In [ ]:
# 2. Đánh giá Ma trận Nhầm lẫn Confusion Matrix của mô hình tối ưu nhất (best.pt)
best_ckpt_path = Path(cfg.checkpoint_dir) / "best.pt"

if best_ckpt_path.exists() and len(trainer.val_loader.dataset) > 0:
    print(f"[*] Đang tải checkpoint tối ưu nhất: {best_ckpt_path}")
    best_ckpt = torch.load(best_ckpt_path, map_location=trainer.device)
    trainer.model.load_state_dict(best_ckpt["model_state_dict"])
    trainer.model.eval()

    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in tqdm(trainer.val_loader, desc="Đánh giá mô hình best.pt"):
            frames_t, labels, seq_lens, _ = batch
            if frames_t.size(0) == 0:
                continue
            frames_t = frames_t.to(trainer.device)
            labels = labels.to(trainer.device)
            seq_lens = seq_lens.to(trainer.device)

            with torch.amp.autocast(device_type=trainer.device.type, enabled=cfg.amp and trainer.device.type == "cuda"):
                p3, p4, p5 = trainer.extractor(frames_t)
                logits = trainer.model((p3, p4, p5), seq_lens=seq_lens)
                _, preds = trainer.compute_loss_and_preds(logits, labels)

            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)

    # In Báo cáo phân loại
    target_names = ["0_Alert (Tỉnh táo)", "1_Drowsy (Buồn ngủ)"]
    print("\n" + "=" * 60)
    print("BÁO CÁO PHÂN LOẠI CHI TIẾT TRÊN TẬP VALIDATION:")
    print("=" * 60)
    print(classification_report(all_targets, all_preds, target_names=target_names, digits=4))

    # Vẽ Confusion Matrix Heatmap
    cm = confusion_matrix(all_targets, all_preds)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=target_names, yticklabels=target_names)
    plt.title("Ma trận Nhầm lẫn (Confusion Matrix) — Checkpoint best.pt")
    plt.xlabel("Nhãn Dự đoán (Predicted)")
    plt.ylabel("Nhãn Thực tế (Ground Truth)")
    cm_path = Path("/kaggle/working/confusion_matrix.png")
    plt.savefig(cm_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[✓] Đã xuất ảnh Ma trận Nhầm lẫn tại: {cm_path}")
else:
    print("[!] Không tìm thấy checkpoint best.pt hoặc tập Validation rỗng.")


## 📦 Section 9: Đóng gói Checkpoints & Tải về Máy tính (Export Artifacts)
Tự động gom toàn bộ các tệp quan trọng (`best.pt`, `last.pt`, `training_history.csv`, biểu đồ đồ thị) thành 1 file ZIP duy nhất để tải về từ Kaggle Output.


In [ ]:
# Tự động nén các kết quả trọng tâm thành file zip để download từ Kaggle Output
output_zip_path = "/kaggle/working/convgru_training_artifacts.zip"

files_to_pack = [
    Path(cfg.checkpoint_dir) / "best.pt",
    Path(cfg.checkpoint_dir) / "last.pt",
    Path(cfg.history_csv_path),
    Path("/kaggle/working/training_curves.png"),
    Path("/kaggle/working/confusion_matrix.png")
]

print(f"[*] Đang đóng gói kết quả vào: {output_zip_path}")
with zipfile.ZipFile(output_zip_path, mode="w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for f in files_to_pack:
        if f.exists():
            zipf.write(f, arcname=f.name)
            print(f" [✓] Đã thêm: {f.name} ({f.stat().st_size / 1024:.1f} KB)")
        else:
            print(f" [!] Bỏ qua (không tồn tại): {f.name}")

if Path(output_zip_path).exists():
    zip_size_mb = Path(output_zip_path).stat().st_size / (1024 * 1024)
    print("=" * 60)
    print(f"[✓] ĐÃ TẠO FILE NÉN THÀNH CÔNG: {output_zip_path} ({zip_size_mb:.2f} MB)")
    print("[*] Bạn có thể tải file này trực tiếp từ tab 'Output' bên phải giao diện Kaggle Notebook!")
    print("=" * 60)
else:
    print("[!] Không tạo được file zip.")
